## Fit Samples Notebook

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numdifftools as nd
import numpy as np
import pandas as pd
from scipy.linalg import inv
from scipy.optimize import curve_fit, minimize
from scipy.stats import lognorm, nbinom

### Get the Data

This gets the data and prints some facts about it.

In [ ]:
results_list = []

for i in range(0, 9001, 1000):
    results_list.append(
        pd.read_csv(
            f"../data/output_data/en_wiki/"
            f"step_list/en_wiki_step_list_intfloat_multilingual-e5-large_{i:d}.csv",
            sep="\t",
        )
    )

results_df = pd.concat(results_list)

In [ ]:
results_df.head(3)

In [ ]:
sucessful_paths = results_df["num_steps"][(results_df["found_target"])]
min_steps = results_df["optimal_steps"][(results_df["found_target"])]
print(f"It was sucessful {sucessful_paths.size/len(results_df):.2%} of the time")

### Fitting the Histograms

In [ ]:
def get_nbinom_log_likelihood(params):
    probs = -nbinom.logpmf(sucessful_paths - min_steps, *params).sum()
    turc = nbinom.logcdf(100 - min_steps, *params).sum()
    return probs + turc

In [ ]:
nbinom_results = minimize(
    get_nbinom_log_likelihood,
    x0=[4, 0.5],
    bounds=[(0, 100), (0, 1)],
    method="Nelder-Mead",
)
print(nbinom_results)

In [ ]:
x_p = np.arange(0, 101, 1)
bins = np.arange(0, 101, 1) + 0.5

plt.figure(figsize=(6, 4))

plt.hist(sucessful_paths - min_steps, bins=bins, density=True, alpha=0.5, label="Data")
plt.plot(x_p, nbinom.pmf(x_p, *nbinom_results.x), alpha=0.5, label="Negative Binomial")

plt.xlabel("Number of steps above optimal")
plt.ylabel("Frequency")

plt.title("multilingual-e5-large - 10 000 games")


plt.show()

In [ ]:
hessian_func = nd.Hessian(
    get_nbinom_log_likelihood, base_step=[1e-1, 1e-3], step_ratio=2, num_steps=4
)
hessian_mat = hessian_func(nbinom_results.x)
cov_mat = inv(hessian_mat)

print(nbinom_results.x)
print(np.sqrt(np.diag(cov_mat)))

In [ ]:
len(sucessful_paths) / nbinom.cdf(100 - np.average(min_steps), *nbinom_results.x)

In [ ]:
def get_faction_scuess(if_success):
    total = len(if_success)
    p_sucess = if_success.mean()
    return p_sucess, np.sqrt(p_sucess * (1 - p_sucess) / total)

In [ ]:
get_faction_scuess(results_df["found_target"].values)

In [ ]:
def vals_from_lognorm(xs, mu, scalex, scaley):
    """This function is used to generate the y values for a lognormal distribution."""
    return scaley / scalex * lognorm.pdf(xs / scalex, mu)

In [ ]:
Y, bins = np.histogram(sucessful_paths, bins=np.arange(0, 101, 1) + 0.5)
X = (bins[:-1] + bins[1:]) / 2

In [ ]:
fit_res = curve_fit(
    vals_from_lognorm, X, Y, [1, 1, 100], bounds=([0, 0, 1], [2, 100, 5000])
)

print(
    f"The spread metric of the lognormal distribution is "
    f"{fit_res[0][0]:.4g} +/- {np.sqrt(fit_res[1][0][0]):.3g}"
)

print(
    f"The median steps of the lognormal distribution is "
    f"{fit_res[0][1]:.4g} +/- {np.sqrt(fit_res[1][1][1]):.3g} "
    f"steps per game."
)

print(
    f"The total area of the lognormal distribution is "
    f"{fit_res[0][2]:.4g} +/- {np.sqrt(fit_res[1][2][2]):.3g} "
    f"successful games at infinity."
)

In [ ]:
x_p = np.arange(0.5, 101, 0.5)

plt.figure(figsize=(6, 4))

plt.hist(sucessful_paths, bins=bins)
plt.plot(
    x_p,
    vals_from_lognorm(x_p, fit_res[0][0], fit_res[0][1], fit_res[0][2]),
    "--k",
    lw=3,
    label="Lognormal fit",
)

plt.xlabel("Number of steps")
plt.ylabel("Frequency")

plt.title("multilingual-e5-large - 10 000 games")

plt.legend(
    title=(
        f"Median - {fit_res[0][1]:.4g} +/- {np.sqrt(fit_res[1][1][1]):.3g}\n"
        f"Total - {fit_res[0][2]:.4g} +/- {np.sqrt(fit_res[1][2][2]):.3g}\n"
        f"Spread - {fit_res[0][0]:.3g} +/- {np.sqrt(fit_res[1][0][0]):.2g}"
    )
)

plt.show()